In [1]:
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import numpy as np

In [122]:
# Read the file containing elements from all models
df_maven = pd.read_csv('input/maven_all_elements.csv',na_filter=False)
len(df_maven)

9501

In [123]:
df_maven.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message'],
      dtype='object')

In [124]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If not space, digits, uppercase letters, lowercase letters, or underscore, ignore it
        if not (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=''
        else:
            new_name+=ch
    return new_name.strip()

In [125]:
print(clean_up('Subject’s Sex'))
print(clean_up(' race_ethnicity '))

Subjects Sex
race_ethnicity


In [126]:
# Clean up column names in both data sets
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [127]:
# Checking for duplicates in questions
duplicates = df_maven[df_maven.duplicated(['Question'],keep=False)]

In [128]:
duplicates.to_csv('output/duplicates.csv',index=False)
len(duplicates)

3125

In [ ]:
# Keeping all duplicate data elements because they are from different models
#df_maven = df_maven.drop_duplicates(['Question'])
#len(df_maven)

7806

In [129]:
df_mmg = pd.read_csv('input/mmg_elements.csv',na_filter=False)
len(df_mmg)

66

In [130]:
df_mmg.columns

Index(['Data Element (DE) Name', 'DE Identifier Sent in HL7 Message',
       'DE Code System', 'Data Element Description', 'Data Type',
       'CDC Priority', 'May Repeat', 'Value Set Name (VADS Hyperlink)',
       'Value Set Code'],
      dtype='object')

In [131]:
df_mmg.loc[:,'Data Element (DE) Name'] = [clean_up(de) for de in df_mmg.loc[:,'Data Element (DE) Name']]

In [132]:
# Checking for duplicate questions
df_mmg[df_mmg.duplicated(['Data Element (DE) Name'])]

,Data Element (DE) Name,DE Identifier Sent in HL7 Message,DE Code System,Data Element Description,Data Type,CDC Priority,May Repeat,Value Set Name (VADS Hyperlink),Value Set Code


In [133]:
# Create an empty co-occurrence matrix
matrix = np.zeros((len(df_maven),len(df_mmg)),dtype=int)
matrix

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [134]:
df_maven.rename(columns={'Question':'Maven_DE'},inplace=True)
df_mmg.rename(columns={'Data Element (DE) Name':'MMG_DE'},inplace=True)

In [135]:

df_empty=pd.DataFrame(matrix,index=df_maven.loc[:,'Maven_DE'],columns=df_mmg.loc[:,'MMG_DE'])

In [136]:
df_empty

MMG_DE,Birth Date,Subjects Sex,Race Category,Other Race Text,Ethnic Group,Country of Birth,Other Birth Place,Country of Usual Residence,Subject Address County,Subject Address State,...,Earliest Date Reported to State,MMWR Week,MMWR Year,Date CDC Was First Verbally Notified of This Case,Date First Reported to PHD,Reporting State,Reporting County,National Reporting Jurisdiction,Comment,Local Subject ID
Maven_DE,,,,,,,,,,,,,,,,,,,,,
DISEASE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_ARBO,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_BT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_COVID,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_GEN,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DATE_SENT_CDC,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SURRG_CONTACT_FLAG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SURRG_PARTNER_FLAG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [137]:
# Compare fuzz score for every column which are data elements from df_mmg

for col in df_empty.columns:
    # Using fuzz.token_set_ratio() - ignores duplicate words and order
    df_empty.loc[:,col] = [fuzz.token_set_ratio(str.replace(maven_de,'_',' '),col) for maven_de in df_empty.index]


In [138]:
df_empty

MMG_DE,Birth Date,Subjects Sex,Race Category,Other Race Text,Ethnic Group,Country of Birth,Other Birth Place,Country of Usual Residence,Subject Address County,Subject Address State,...,Earliest Date Reported to State,MMWR Week,MMWR Year,Date CDC Was First Verbally Notified of This Case,Date First Reported to PHD,Reporting State,Reporting County,National Reporting Jurisdiction,Comment,Local Subject ID
Maven_DE,,,,,,,,,,,,,,,,,,,,,
DISEASE,35,42,30,27,11,9,33,18,28,36,...,32,25,25,21,30,36,9,21,14,35
ALL_ARBO,22,20,29,17,30,25,16,12,20,28,...,26,12,35,18,24,17,25,26,13,33
ALL_BT,25,33,21,29,11,18,17,12,29,30,...,22,13,13,15,19,19,18,22,15,45
ALL_COVID,21,19,27,17,19,24,15,23,26,20,...,25,11,11,21,23,17,24,25,25,32
ALL_GEN,24,21,30,27,21,17,17,18,21,21,...,21,25,25,14,18,18,26,26,29,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DATE_SENT_CDC,57,40,46,43,16,28,33,31,40,53,...,47,27,18,76,47,29,34,36,40,48
SURRG_CONTACT_FLAG,21,20,39,30,33,35,29,41,35,31,...,29,7,22,27,32,30,47,41,32,35
SURRG_PARTNER_FLAG,29,27,39,30,27,29,29,36,30,41,...,37,15,22,30,32,36,35,33,8,29


In [139]:
df_empty.to_csv('output/maven_mmg_scores.csv')

In [ ]:
# Check for every column (mmg element) if there is a close match
outfile = open('output/mmg_matches.csv','w')
outfile.write('mmg_element,match_list\n')
for col in df_empty.columns:
    df_sub = df_empty[df_empty[col]>=80]
    if len(df_sub)==0:
        print("No close matches for",col)
        outfile.write(col+','+'\n')
    else:
        #print(df_empty.loc[df_empty[col].idxmax()])
        # print the matches
        #print("Matches for",col,list(df_sub.index))
        outfile.write(col+','+str(list(df_sub.index))+'\n')
outfile.close()
            
        

No close matches for Subjects Sex
No close matches for Ethnic Group
No close matches for Subject Died
No close matches for Condition Code
No close matches for Local Record ID
No close matches for Immediate National Notifiable Condition
No close matches for Person Reporting to CDC  Email
No close matches for Date First Electronically Submitted
No close matches for Date CDC Was First Verbally Notified of This Case
No close matches for Local Subject ID


In [141]:
# Compare CSTE elements against Maven
# Read CSTE elements
df_cste = pd.read_csv('input/cste_elements.csv',na_filter=False)
len(df_cste)

62

In [142]:
df_cste.columns

Index(['cste_de', 'Data Class'], dtype='object')

In [143]:
df_cste.loc[:,'cste_de'] = [clean_up(de) for de in df_cste.loc[:,'cste_de']]

In [144]:
# Find any duplicates
df_cste[df_cste.duplicated(['cste_de'])]

,cste_de,Data Class


In [145]:
# Create an empty co-occurrence matrix
matrix = np.zeros((len(df_maven),len(df_cste)),dtype=int)
matrix

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [147]:
df_empty=pd.DataFrame(matrix,index=df_maven.loc[:,'Maven_DE'],columns=df_cste.loc[:,'cste_de'])

In [148]:
df_empty

cste_de,Discharge Date,Sex Assigned at Birth,Date of Death,Pregnancy Status,Person Deceased,Hospitalized,Admission Date,Sexual Orientation,Gender Identity,Sex,...,Estimated Date of Delivery EDD,International Travel,Domestic Travel,Specify Different Travel Exposure Window,International Destinations of Recent Travel,Domestic Destinations of Recent Travel,Reasons for Travel,Date of Arrival to Travel Destination,Date of Departure from Travel Destination,Cause of Death
Maven_DE,,,,,,,,,,,,,,,,,,,,,
DISEASE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_ARBO,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_BT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_COVID,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ALL_GEN,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DATE_SENT_CDC,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SURRG_CONTACT_FLAG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SURRG_PARTNER_FLAG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [149]:
# Compare fuzz score for every column which are data elements from df_cste

for col in df_empty.columns:
    # Using fuzz.token_set_ratio() - ignores duplicate words and order
    df_empty.loc[:,col] = [fuzz.token_set_ratio(str.replace(maven_de,'_',' '),col) for maven_de in df_empty.index]


In [150]:
df_empty

cste_de,Discharge Date,Sex Assigned at Birth,Date of Death,Pregnancy Status,Person Deceased,Hospitalized,Admission Date,Sexual Orientation,Gender Identity,Sex,...,Estimated Date of Delivery EDD,International Travel,Domestic Travel,Specify Different Travel Exposure Window,International Destinations of Recent Travel,Domestic Destinations of Recent Travel,Reasons for Travel,Date of Arrival to Travel Destination,Date of Departure from Travel Destination,Cause of Death
Maven_DE,,,,,,,,,,,,,,,,,,,,,
DISEASE,48,36,30,26,45,32,48,40,27,40,...,27,30,36,26,24,27,32,18,21,29
ALL_ARBO,36,28,38,25,35,20,27,23,9,0,...,26,29,17,17,16,13,23,22,20,36
ALL_BT,20,30,32,27,19,22,30,17,19,0,...,22,31,19,13,16,14,25,19,13,30
ALL_COVID,26,20,27,16,25,38,26,15,25,0,...,26,28,17,20,19,21,22,22,16,26
ALL_GEN,38,21,30,17,36,32,29,24,27,20,...,22,30,18,17,20,18,24,23,17,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DATE_SENT_CDC,59,41,62,34,50,16,59,32,43,25,...,47,24,36,26,29,31,32,47,47,44
SURRG_CONTACT_FLAG,38,31,32,35,24,27,25,44,18,10,...,25,37,30,24,33,32,28,25,27,31
SURRG_PARTNER_FLAG,44,41,32,35,36,20,31,33,30,10,...,25,26,18,28,26,25,33,29,37,31


In [151]:
df_empty.to_csv('output/maven_cste_scores.csv')

In [152]:
# Check for every column (cste element) if there is a close match
outfile = open('output/cste_matches.csv','w')
outfile.write('cste_element,match_list\n')
for col in df_empty.columns:
    df_sub = df_empty[df_empty[col]>=80]
    if len(df_sub)==0:
        print("No close matches for",col)
        outfile.write(col+',''\n')
    else:
        #print(df_empty.loc[df_empty[col].idxmax()])
        # print the matches
        #print("Matches for",col,list(df_sub.index))
        outfile.write(col+','+str(list(df_sub.index))+'\n')
outfile.close()

No close matches for Death Associated with Condition
No close matches for Educational Attainment
No close matches for Disability Type
No close matches for Disability Type Indicator
No close matches for Summative Risk Factor Indicator
No close matches for Summative Underlying Health Condition Indicator
No close matches for Vaccination Doses Prior to Illness Onset
No close matches for Units of Measure
No close matches for DateTime of Lab Result
